# Earthquakes Silver Layer

Transforms USGS GeoJSON earthquake feed snapshots from `weather_api.bronze.earthquakes_raw` into clean, typed, deduplicated Silver tables.

**Tables created:**
- `weather_api.silver.earthquakes` — Latest version of each event (grain: one row per `event_id`). Deduplicated by keeping the event with the latest `updated_at` timestamp; ties broken by latest `snapshot_at`.
- `weather_api.silver.earthquakes_event_history` — All event versions across feed snapshots (grain: one row per `event_id` + `snapshot_at`). Preserves how event details evolve over time (magnitude revisions, status changes from automatic → reviewed).

**Processing:** Batch read from Bronze → explode features → extract typed fields → Delta MERGE. Idempotent and safe for reruns.

In [0]:
# ════════════════════════════════════════════════════════════════════
# EARTHQUAKES SILVER — TRANSFORM
# ════════════════════════════════════════════════════════════════════
from pyspark.sql.functions import (
    col, explode, to_timestamp, from_unixtime, lit, when, expr,
    count as cnt, countDistinct, desc, max as max_fn, min as min_fn,
    row_number
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# ─── Schema & Config ────────────────────────────────────────────────
spark.sql("CREATE SCHEMA IF NOT EXISTS weather_api.silver")

BRONZE_TABLE        = "weather_api.bronze.earthquakes_raw"
SILVER_EVENTS       = "weather_api.silver.earthquakes"
SILVER_EVENT_HIST   = "weather_api.silver.earthquakes_event_history"

# ─── Read Bronze Source ────────────────────────────────────────────
bronze_eq = spark.table(BRONZE_TABLE)
total_bronze_rows = bronze_eq.count()
print(f"Bronze rows (feed snapshots): {total_bronze_rows}")

# ─── Transform: Explode Features & Extract Fields ─────────────────
# Each bronze row is a complete GeoJSON FeatureCollection with an array
# of earthquake features. We explode into one row per event and extract
# all fields from the nested struct into typed columns.

# Epoch millisecond timestamps are converted to Spark timestamps by
# dividing by 1000 (ms → s) and using from_unixtime.
silver_eq = bronze_eq.select(
    # Lineage
    col("_source_file_path"),
    col("_ingest_timestamp"),
    col("_rescued_data"),
    to_timestamp(col("metadata.retrieval_timestamp_utc")).alias("snapshot_at"),
    col("metadata.api_endpoint"),
    col("metadata.source"),
    # Feed-level metadata from USGS
    col("raw_response.metadata.api").alias("feed_api_version"),
    col("raw_response.metadata.title").alias("feed_title"),
    (from_unixtime(col("raw_response.metadata.generated") / 1000))
        .cast("timestamp").alias("feed_generated_at"),
    # Explode the features array
    explode(col("raw_response.features")).alias("f"),
).select(
    # ── Lineage ──
    col("_source_file_path").alias("source_file_path"),
    col("_ingest_timestamp").alias("ingested_at"),
    col("_rescued_data"),
    col("snapshot_at"),
    col("api_endpoint"),
    col("source"),
    col("feed_api_version"),
    col("feed_title"),
    col("feed_generated_at"),
    # ── Event Identity ──
    col("f.id").alias("event_id"),
    col("f.properties.code").alias("event_code"),
    # ── Measurements ──
    col("f.properties.mag").alias("magnitude"),
    col("f.properties.magType").alias("magnitude_type"),
    col("f.properties.place").alias("place_description"),
    col("f.properties.title").alias("event_title"),
    col("f.properties.type").alias("event_type"),
    col("f.properties.status").alias("review_status"),
    col("f.properties.alert").alias("alert_level"),
    col("f.properties.sig").alias("significance"),
    col("f.properties.tsunami").cast("boolean").alias("tsunami_flag"),
    col("f.properties.net").alias("seismic_network"),
    col("f.properties.nst").alias("station_count"),
    col("f.properties.gap").alias("azimuthal_gap"),
    col("f.properties.dmin").alias("distance_to_nearest_station_deg"),
    col("f.properties.rms").alias("rms_travel_time_residual"),
    col("f.properties.felt").alias("felt_report_count"),
    col("f.properties.cdi").alias("community_intensity"),
    col("f.properties.mmi").alias("mercalli_intensity"),
    col("f.properties.tz").alias("timezone_offset"),
    # ── URLs ──
    col("f.properties.url").alias("event_page_url"),
    col("f.properties.detail").alias("detail_api_url"),
    # ── ID lists (comma-delimited → ARRAY<STRING>, empty elements removed) ──
    expr("filter(split(f.properties.ids, ','), x -> x != '')").alias("all_event_ids"),
    expr("filter(split(f.properties.sources, ','), x -> x != '')").alias("all_sources"),
    expr("filter(split(f.properties.types, ','), x -> x != '')").alias("available_product_types"),
    # ── Timestamps (epoch ms → UTC timestamp) ──
    (from_unixtime(col("f.properties.time") / 1000))
        .cast("timestamp").alias("event_time"),
    (from_unixtime(col("f.properties.updated") / 1000))
        .cast("timestamp").alias("updated_at"),
    # ── Geometry ──
    col("f.geometry.coordinates")[0].alias("longitude"),
    col("f.geometry.coordinates")[1].alias("latitude"),
    col("f.geometry.coordinates")[2].alias("depth_km"),
    col("f.geometry.type").alias("geometry_type"),
)

total_events = silver_eq.count()
unique_events = silver_eq.select(countDistinct("event_id")).collect()[0][0]
print(f"Total event rows (pre-dedup): {total_events}")
print(f"Unique event IDs: {unique_events}")

# ─── Validation ────────────────────────────────────────────────────
print("\n--- Null counts (pre-dedup) ---")
silver_eq.agg(
    cnt("*").alias("total"),
    cnt("event_id").alias("id_nn"),
    cnt("magnitude").alias("mag_nn"),
    cnt("event_time").alias("time_nn"),
    cnt("updated_at").alias("updated_nn"),
    cnt("longitude").alias("lon_nn"),
    cnt("latitude").alias("lat_nn"),
    cnt("depth_km").alias("depth_nn"),
    cnt("place_description").alias("place_nn"),
    cnt("review_status").alias("status_nn"),
    cnt("event_type").alias("type_nn"),
    cnt("magnitude_type").alias("magType_nn"),
    cnt("alert_level").alias("alert_nn"),
    cnt("tsunami_flag").alias("tsunami_nn"),
).show(truncate=False)

# Validate coordinate ranges
invalid_coords = silver_eq.filter(
    (col("longitude") < -180) | (col("longitude") > 180) |
    (col("latitude") < -90) | (col("latitude") > 90)
)
invalid_coord_count = invalid_coords.count()
print(f"Invalid coordinates: {invalid_coord_count}")

# Validate magnitude range
invalid_mag = silver_eq.filter(
    (col("magnitude") < -2) | (col("magnitude") > 12)
)
invalid_mag_count = invalid_mag.count()
print(f"Implausible magnitudes: {invalid_mag_count}")

# Rescued data
rescued = bronze_eq.filter(col("_rescued_data").isNotNull()).count()
print(f"Rescued data records: {rescued}")

# ─── Deduplicate: Latest Version Per Event ─────────────────────────
# The USGS past-day feed snapshots overlap, so the same event appears
# in multiple snapshots. We keep the version with the latest updated_at
# timestamp. Ties broken by latest snapshot_at (deterministic).
dedup_window = Window.partitionBy("event_id").orderBy(
    col("updated_at").desc(),
    col("snapshot_at").desc()
)
silver_eq_latest = (
    silver_eq
    .withColumn("_rn", row_number().over(dedup_window))
    .filter(col("_rn") == 1)
    .drop("_rn")
)

latest_count = silver_eq_latest.count()
print(f"\nDeduplicated (latest per event): {latest_count}")
assert latest_count == unique_events, \
    f"Dedup mismatch: {latest_count} != {unique_events}"

print("\n✓ Transform validation passed.")

In [0]:
# ════════════════════════════════════════════════════════════════════
# EARTHQUAKES SILVER — WRITE & VALIDATE
# ════════════════════════════════════════════════════════════════════
# Uses Delta MERGE for idempotent upserts. Safe for reruns.

# ─── Write: Primary Earthquakes Table ─────────────────────────────
# One row per event_id (latest version). MERGE on event_id.
# Schema fix: drop if all_event_ids is string (old type → ARRAY<STRING>)
if spark.catalog.tableExists(SILVER_EVENTS):
    _ids_type = spark.sql(f"DESCRIBE TABLE {SILVER_EVENTS}").filter("col_name = 'all_event_ids'").collect()[0].data_type
    if _ids_type == "string":
        spark.sql(f"DROP TABLE {SILVER_EVENTS}")
        print(f"Dropped {SILVER_EVENTS} for array conversion")

if spark.catalog.tableExists(SILVER_EVENTS):
    DeltaTable.forName(spark, SILVER_EVENTS).alias("t").merge(
        silver_eq_latest.alias("s"),
        "t.event_id = s.event_id"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    print(f"MERGE completed: {SILVER_EVENTS}")
else:
    silver_eq_latest.write.format("delta").saveAsTable(SILVER_EVENTS)
    print(f"Created: {SILVER_EVENTS}")

# ─── Write: Event History Table ───────────────────────────────────
# All versions across snapshots. MERGE on (event_id, snapshot_at).
# Schema fix: drop if all_event_ids is string (old type → ARRAY<STRING>)
if spark.catalog.tableExists(SILVER_EVENT_HIST):
    _hist_ids_type = spark.sql(f"DESCRIBE TABLE {SILVER_EVENT_HIST}").filter("col_name = 'all_event_ids'").collect()[0].data_type
    if _hist_ids_type == "string":
        spark.sql(f"DROP TABLE {SILVER_EVENT_HIST}")
        print(f"Dropped {SILVER_EVENT_HIST} for array conversion")

if spark.catalog.tableExists(SILVER_EVENT_HIST):
    DeltaTable.forName(spark, SILVER_EVENT_HIST).alias("t").merge(
        silver_eq.alias("s"),
        "t.event_id = s.event_id AND t.snapshot_at = s.snapshot_at"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    print(f"MERGE completed: {SILVER_EVENT_HIST}")
else:
    silver_eq.write.format("delta").saveAsTable(SILVER_EVENT_HIST)
    print(f"Created: {SILVER_EVENT_HIST}")

# ─── Post-Write Validation ─────────────────────────────────────────
print("\n" + "=" * 70)
print("POST-WRITE VALIDATION")
print("=" * 70)

# Primary table
result = spark.table(SILVER_EVENTS)
result_count = result.count()
result_keys = result.select(countDistinct("event_id")).collect()[0][0]
print(f"\n[{SILVER_EVENTS}]")
print(f"  Rows: {result_count}")
print(f"  Distinct event IDs: {result_keys}")
result_dups = result.groupBy("event_id").agg(cnt("*").alias("n")).filter(col("n") > 1)
print(f"  Duplicate event IDs: {result_dups.count()}")
assert result_count == result_keys, "Duplicate event IDs in primary table!"

print("  Null counts:")
result.agg(
    cnt("*").alias("total"),
    cnt("event_id").alias("id_nn"),
    cnt("magnitude").alias("mag_nn"),
    cnt("event_time").alias("time_nn"),
    cnt("longitude").alias("lon_nn"),
    cnt("latitude").alias("lat_nn"),
    cnt("depth_km").alias("depth_nn"),
    cnt("place_description").alias("place_nn"),
    cnt("review_status").alias("status_nn"),
    cnt("alert_level").alias("alert_nn"),
    cnt("tsunami_flag").alias("tsunami_nn"),
).show(truncate=False)

print("  Timestamp ranges:")
result.agg(
    min_fn("event_time").alias("min_event_time"),
    max_fn("event_time").alias("max_event_time"),
    min_fn("updated_at").alias("min_updated"),
    max_fn("updated_at").alias("max_updated"),
    min_fn("snapshot_at").alias("min_snapshot"),
    max_fn("snapshot_at").alias("max_snapshot"),
).show(truncate=False)

# History table
hist = spark.table(SILVER_EVENT_HIST)
hist_count = hist.count()
hist_keys = hist.select(countDistinct("event_id", "snapshot_at")).collect()[0][0]
print(f"\n[{SILVER_EVENT_HIST}]")
print(f"  Rows: {hist_count}")
print(f"  Distinct (event_id, snapshot_at): {hist_keys}")
hist_dups = hist.groupBy("event_id", "snapshot_at").agg(cnt("*").alias("n")).filter(col("n") > 1)
print(f"  Duplicate keys: {hist_dups.count()}")
assert hist_count == hist_keys, "Duplicate keys in history table!"

# Reconciliation
print("\n--- Reconciliation ---")
print(f"  Bronze feed snapshots: {total_bronze_rows}")
print(f"  Total event rows (pre-dedup): {total_events}")
print(f"  Unique event IDs: {unique_events}")
print(f"  Silver primary rows: {result_count} (should = unique events)")
print(f"  Silver history rows: {hist_count} (should = total events)")
print(f"  Invalid coordinates: {invalid_coord_count}")
print(f"  Implausible magnitudes: {invalid_mag_count}")
print(f"  Rescued data: {rescued}")
assert result_count == unique_events, "Primary table count mismatch!"
assert hist_count == total_events, "History table count mismatch!"
assert invalid_coord_count == 0, "Invalid coordinates found!"

# Sample rows
print("\n--- Top 5 events by magnitude ---")
result.select(
    "event_id", "magnitude", "magnitude_type", "place_description",
    "event_time", "review_status", "alert_level", "tsunami_flag",
    "longitude", "latitude", "depth_km"
).orderBy(desc("magnitude")).show(5, truncate=False)

print("\n--- Event type distribution ---")
result.groupBy("event_type").agg(cnt("*").alias("cnt")).orderBy(desc("cnt")).show()

print("\n--- Review status distribution ---")
result.groupBy("review_status").agg(cnt("*").alias("cnt")).orderBy(desc("cnt")).show()

print("\n--- Sample event history (events appearing in multiple snapshots) ---")
hist.groupBy("event_id").agg(
    cnt("*").alias("versions"),
    max_fn("magnitude").alias("max_mag"),
    min_fn("magnitude").alias("min_mag"),
    max_fn("review_status").alias("latest_status")
).filter(col("versions") > 1).orderBy(desc("versions")).show(5, truncate=False)

print("\n✓ Earthquakes Silver layer complete.")

In [0]:
# ════════════════════════════════════════════════════════════════════
# EARTHQUAKES SILVER — COLUMN COMMENTS
# ════════════════════════════════════════════════════════════════════
# Idempotent: ALTER COLUMN COMMENT can be run multiple times safely.

_eq_comments = {
    "source_file_path": "UC Volume path of the source JSON file",
    "ingested_at": "Timestamp when Auto Loader ingested this record into Bronze",
    "_rescued_data": "Fields that did not match the Bronze schema (null = no schema drift)",
    "snapshot_at": "UTC timestamp when the USGS feed snapshot was retrieved",
    "api_endpoint": "USGS feed API URL",
    "source": "Pipeline source label",
    "feed_api_version": "USGS feed API version",
    "feed_title": "USGS feed title",
    "feed_generated_at": "UTC timestamp when USGS generated this feed (epoch ms → timestamp)",
    "event_id": "USGS unique event identifier (primary business key)",
    "event_code": "Event code without the network prefix",
    "magnitude": "Earthquake magnitude (Richter or moment, depending on magnitude_type)",
    "magnitude_type": "Magnitude type: ml (local), md (duration), mb (body wave), mww (moment), mb_lg",
    "place_description": "Human-readable location description",
    "event_title": "Full event title with magnitude and place",
    "event_type": "Event type: earthquake, explosion, etc.",
    "review_status": "Review status: automatic (not yet reviewed) or reviewed",
    "alert_level": "Alert level: green, yellow, orange, red, or null",
    "significance": "Composite significance score (0–1000)",
    "tsunami_flag": "True if a tsunami warning was issued for this event",
    "seismic_network": "Seismic network that reported the event (ak, ci, hv, nn, us, etc.)",
    "station_count": "Number of seismic stations used to locate the event",
    "azimuthal_gap": "Largest azimuthal angle between stations as seen from the epicentre, in degrees",
    "distance_to_nearest_station_deg": "Horizontal distance to the nearest reporting station, in degrees",
    "rms_travel_time_residual": "Root-mean-square travel-time residual (quality of fit)",
    "felt_report_count": "Number of citizen 'Did You Feel It' reports",
    "community_intensity": "Community Did-You-Feel-It intensity (1–10)",
    "mercalli_intensity": "Modified Mercalli Intensity (instrumental, I–XII)",
    "timezone_offset": "Timezone offset at the epicentre (often null in USGS data)",
    "event_page_url": "USGS event page URL",
    "detail_api_url": "USGS detail API endpoint URL",
    "all_event_ids": "Array of all associated event IDs across networks",
    "all_sources": "Array of seismic network source codes",
    "available_product_types": "Array of available USGS product types for this event",
    "event_time": "Earthquake origin time, as a UTC timestamp (epoch ms → timestamp)",
    "updated_at": "Last updated time, as a UTC timestamp (epoch ms → timestamp)",
    "longitude": "Epicentre longitude, in degrees",
    "latitude": "Epicentre latitude, in degrees",
    "depth_km": "Depth of the earthquake below the surface, in kilometres",
    "geometry_type": "GeoJSON geometry type (always Point)",
}

_comment_count = 0
for _table in [SILVER_EVENTS, SILVER_EVENT_HIST]:
    for _col, _comment in _eq_comments.items():
        _escaped = _comment.replace("'", "''")
        spark.sql(f"ALTER TABLE {_table} ALTER COLUMN {_col} COMMENT '{_escaped}'")
        _comment_count += 1
print(f"Applied {_comment_count} column comments across 2 earthquake Silver tables.")